In [5]:
import ezomero
import tqdm
import shutil
import os
from skimage import io
from ultralytics import YOLO
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import torch
torch.cuda.is_available()

WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Johan\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


True

In [12]:
host = 'omero-int.biotec.tu-dresden.de'
group = 'Grapin-Botton Lab'
port = 4064
secure = True
user='johamuel'

conn = ezomero.connect(host=host, port=port, secure=secure, user=user, group=group)

In [3]:
dataset_id = 409

In [13]:
working_directory = r'E:\Data\BiAPoL\yolo_demo\dataset'  # the directory where you want to save the dataset - replace with your own directory
images_dir = os.path.join(working_directory, 'images')
labels_dir = os.path.join(working_directory, 'labels')
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

In [22]:
object_classes = {
    'cell': 0,
    'spheroid': 1,
}

annotation_tag_id = 19762

In [26]:
img_ids = ezomero.get_image_ids(conn, dataset=dataset_id)  # replace 259 with the id of the dataset you want to use
img_ids = [i for i in img_ids if len(ezomero.get_tag_ids(conn, object_type='Image', object_id=i, ns=None)) > 0]  # filter images that have tags

In [29]:
for img_id in tqdm.tqdm(img_ids):

    # get the image and metadata and save the image locally
    metadata, image = ezomero.get_image(conn, image_id=img_id, dim_order='tczyx')
    image_filename = os.path.join(working_directory, 'images', f'{metadata.name}.png')
    labels_filename = os.path.join(working_directory, 'labels', f'{metadata.name}.txt')
    
    image = image.squeeze()  # remove singleton dimensions (TCZ)
    io.imsave(image_filename, image.squeeze())

    # determine the width and height of the image and get associated rois
    width, height = image.shape[1], image.shape[0]
    roi_ids = ezomero.get_roi_ids(conn, image_id=img_id)

    shapes = []
    for roi_id in roi_ids:
        shape_ids = ezomero.get_shape_ids(conn, roi_id=roi_id)
        for shape_id in shape_ids:
            shapes.append(ezomero.get_shape(conn, shape_id=shape_id))

    # now to convert the shapes to YOLO format
    with open(labels_filename, 'w') as f:
        for shape in shapes:
            w = shape.width / width
            h = shape.height / height

            x = shape.x / width + w / 2
            y = shape.y / height + h / 2

            class_id = object_classes[shape.label.lower()]

            f.write(f'{class_id} {x} {y} {w} {h}\n')

100%|██████████| 64/64 [04:41<00:00,  4.40s/it]


In [30]:
# Output directories
output_dirs = {
    'train': {'images': os.path.join(working_directory, 'train/images'), 'labels': os.path.join(working_directory, 'train/labels')},
    'val': {'images': os.path.join(working_directory, 'val/images'), 'labels': os.path.join(working_directory, 'val/labels')},
    'test': {'images': os.path.join(working_directory, 'test/images'), 'labels': os.path.join(working_directory, 'test/labels')}
}

# Create output directories if they don't exist
for key in output_dirs:
    os.makedirs(output_dirs[key]['images'], exist_ok=True)
    os.makedirs(output_dirs[key]['labels'], exist_ok=True)

In [31]:
# Get list of all image files
image_files = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]

# Split the dataset into train, val, and test
train_files, test_files = train_test_split(image_files, test_size=0.30, random_state=42)
val_files, test_files = train_test_split(test_files, test_size=1/3, random_state=42)  # 1/3 of 30% => 10%

# Move the files to the respective directories
for key, files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
    for file in files:
        image_file = os.path.join(images_dir, file)
        label_file = os.path.join(labels_dir, file.replace('.png', '.txt'))

        shutil.copy(image_file, output_dirs[key]['images'])
        shutil.copy(label_file, output_dirs[key]['labels'])

In [32]:
print('Number of samples in training set:', len(train_files))
print('Number of samples in validation set:', len(val_files))
print('Number of samples in test set:', len(test_files))

Number of samples in training set: 44
Number of samples in validation set: 13
Number of samples in test set: 7


In [33]:
model = YOLO("yolov8n.pt")

100%|██████████| 6.25M/6.25M [00:03<00:00, 1.83MB/s]


In [36]:
model.train(data=r"E:\Data\BiAPoL\yolo_demo\dataset\dataset_config.yaml",
            epochs=100, imgsz=512, device=0, batch=64, flipud=0.5, fliplr=0.5,
            dropout=True, optimizer='Adam', seed=42, plots=True)

Ultralytics 8.3.165  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Data\BiAPoL\yolo_demo\dataset\dataset_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=True, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train22, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspectiv

train: Scanning E:\Data\BiAPoL\yolo_demo\dataset\train\labels.cache... 44 images, 15 backgrounds, 0 corrupt: 100%|██████████| 44/44 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access  (ping: 0.00.0 ms, read: 67.913.7 MB/s, size: 3497.8 KB)


val: Scanning E:\Data\BiAPoL\yolo_demo\dataset\val\labels.cache... 13 images, 8 backgrounds, 0 corrupt: 100%|██████████| 13/13 [00:00<?, ?it/s]


...

                   all         13         20      0.268      0.467      0.336       0.19



100 epochs completed in 0.016 hours.
Optimizer stripped from runs\detect\train22\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train22\weights\best.pt, 6.2MB

Validating runs\detect\train22\weights\best.pt...
Ultralytics 8.3.165  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 20.42it/s]


                   all         13         20      0.268      0.467      0.336       0.19
                  cell          3          5          0          0     0.0323    0.00956
              spheroid          5         15      0.537      0.933       0.64      0.371
Speed: 0.1ms preprocess, 0.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\detect\train22


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000029BCB208470>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0